# Chapter 2: First Query in 10 Minutes

This notebook contains all the code examples from Chapter 2.

## Setup

Make sure you have the required packages installed:
```bash
pip install duckdb polars pyarrow jupyter pandas
```

## Version Check

In [ ]:
import duckdb
print(f'DuckDB {duckdb.__version__}')
# Should see 1.0.0 or higher

## Generate Sample Data (if needed)

Run this cell if you don't have `orders.parquet` yet:

In [3]:
import polars as pl
import datetime

# 10M orders across 2 years
orders = pl.DataFrame({
    'order_id': range(1, 10_000_001),
    'customer_id': (pl.arange(1, 10_000_001, eager=True) % 500_000) + 1,
    'order_date': pl.date_range(
        datetime.date(2022, 1, 1),
        datetime.date(2023, 12, 31),
        interval='1d',
        eager=True
    ).sample(10_000_000, with_replacement=True),
    'amount': (pl.arange(1, 10_000_001, eager=True) % 500 + 20.0),
    'status': pl.Series(['completed', 'pending', 'cancelled', 'refunded'])\
        .sample(10_000_000, with_replacement=True)
})

# Write as Parquet
orders.write_parquet('orders.parquet', compression='snappy')
print(f"[OK] Generated {len(orders):,} rows -> orders.parquet")

[OK] Generated 10,000,000 rows -> orders.parquet


## Your First Query

Query 10M rows of Parquet directly - no import, no loading:

In [4]:
import duckdb

# Query Parquet directly - no import, no loading
con = duckdb.connect()

result = con.execute("""
    SELECT
        DATE_TRUNC('month', order_date) as month,
        status,
        COUNT(*) as orders,
        SUM(amount) as revenue,
        AVG(amount) as avg_order
    FROM read_parquet('orders.parquet')
    WHERE order_date >= '2023-01-01'
    GROUP BY 1, 2
    ORDER BY 1, 2
""").fetchdf()

print(result)

        month     status  orders     revenue   avg_order
0  2023-01-01  cancelled  106530  28789979.0  270.252314
1  2023-01-01  completed  106182  28536724.0  268.752934
2  2023-01-01    pending  106218  28608214.0  269.334896
3  2023-01-01   refunded  106243  28698947.0  270.125533
4  2023-02-01  cancelled   95292  25652692.0  269.200898
5  2023-02-01  completed   95563  25775760.0  269.725312
6  2023-02-01    pending   95689  25840966.0  270.051584
7  2023-02-01   refunded   95994  25880409.0  269.604444
8  2023-03-01  cancelled  106069  28605139.0  269.684253
9  2023-03-01  completed  105893  28566670.0  269.769201
10 2023-03-01    pending  106190  28665872.0  269.948884
11 2023-03-01   refunded  106075  28514766.0  268.817026
12 2023-04-01  cancelled  103087  27711889.0  268.820404
13 2023-04-01  completed  102590  27680801.0  269.819680
14 2023-04-01    pending  102754  27728261.0  269.850916
15 2023-04-01   refunded  102713  27727630.0  269.952489
16 2023-05-01  cancelled  10628

## Check Compression Ratio

In [5]:
import os

file_size = os.path.getsize('orders.parquet') / 1_000_000
print(f"Compressed: {file_size:.0f} MB")
# Uncompressed in memory would be ~1.2 GB

Compressed: 101 MB


## Query Plan

See how DuckDB applies filters before loading data:

In [6]:
plan = con.execute("""
    EXPLAIN
    SELECT DATE_TRUNC('month', order_date) as month,
           COUNT(*) as orders
    FROM read_parquet('orders.parquet')
    WHERE order_date >= '2023-01-01'
    GROUP BY 1
""").fetchall()

for line in plan:
    print(line[0])

physical_plan


## Smoke Test Checklist

In [7]:
import duckdb
import polars as pl
import pyarrow.parquet as pq

# [OK] DuckDB can query Parquet
assert duckdb.execute("SELECT COUNT(*) FROM read_parquet('orders.parquet')").fetchone()[0] == 10_000_000
print("✓ DuckDB can query Parquet")

# [OK] Polars can read Parquet lazily
df = pl.scan_parquet('orders.parquet')
assert df.select(pl.count()).collect().item() == 10_000_000
print("✓ Polars can read Parquet lazily")

# [OK] PyArrow can read metadata without loading data
metadata = pq.read_metadata('orders.parquet')
print(f"✓ PyArrow: {metadata.num_rows:,} rows in {metadata.num_row_groups} row groups")

print("\n🎉 Stack verified. You're ready.")

✓ DuckDB can query Parquet
✓ Polars can read Parquet lazily
✓ PyArrow: 10,000,000 rows in 38 row groups

🎉 Stack verified. You're ready.


/var/folders/cd/l_bbk7dd2fbdr7b_mtcr971m0000gn/T/ipykernel_36096/435845547.py:11: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  assert df.select(pl.count()).collect().item() == 10_000_000


## Quick Win Queries

### Top 10 Customers by Total Spend

In [8]:
top_customers = con.execute("""
    SELECT customer_id,
           SUM(amount) as total_spent,
           COUNT(*) as order_count
    FROM read_parquet('orders.parquet')
    GROUP BY customer_id
    ORDER BY total_spent DESC
    LIMIT 10
""").fetchdf()

print(top_customers)

   customer_id  total_spent  order_count
0       152500      10380.0           20
1       155000      10380.0           20
2       215500      10380.0           20
3       287500      10380.0           20
4       307500      10380.0           20
5        61000      10380.0           20
6       104500      10380.0           20
7       138500      10380.0           20
8       148000      10380.0           20
9       284000      10380.0           20


### Daily Order Volume with 7-Day Moving Average

In [9]:
daily_orders = con.execute("""
    SELECT order_date,
           COUNT(*) as orders,
           AVG(COUNT(*)) OVER (
               ORDER BY order_date
               ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
           ) as rolling_avg_7d
    FROM read_parquet('orders.parquet')
    GROUP BY order_date
    ORDER BY order_date
""").fetchdf()

print(daily_orders.head(10))

  order_date  orders  rolling_avg_7d
0 2022-01-01   13721    13721.000000
1 2022-01-02   13740    13730.500000
2 2022-01-03   13613    13691.333333
3 2022-01-04   13681    13688.750000
4 2022-01-05   13610    13673.000000
5 2022-01-06   13716    13680.166667
6 2022-01-07   13752    13690.428571
7 2022-01-08   13761    13696.142857
8 2022-01-09   13757    13698.571429
9 2022-01-10   13578    13693.571429


### Revenue by Status (Pie Chart Data)

In [10]:
revenue_by_status = con.execute("""
    SELECT status,
           SUM(amount) as revenue,
           ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 2) as pct
    FROM read_parquet('orders.parquet')
    GROUP BY status
    ORDER BY revenue DESC
""").fetchdf()

print(revenue_by_status)

      status      revenue    pct
0    pending  674045846.0  25.01
1   refunded  673804885.0  25.00
2  cancelled  673680152.0  25.00
3  completed  673469117.0  24.99
